# 🧠 RAG Chunking & Embedding Prep
## Document Segmentation, Token Budgeting, and Vector Store Pre-Flight

**Pack ID:** `pack_0006` · **Category:** `rag` · **Difficulty:** `intermediate` · **Version:** `1.1.0`

---
Prepares arbitrary documentation for production Retrieval-Augmented Generation (RAG) with header-aware chunking, token estimation, and mock/semantic embedding generation.


In [ ]:
import json
import time
import hashlib
import re
from pathlib import Path

DOC_PATH = Path('data/documents.txt')
OUT_CHUNKS = Path('outputs/chunks.json')
OUT_REPORT = Path('outputs/rag_preparation_report.md')
OUT_CHUNKS.parent.mkdir(parents=True, exist_ok=True)

raw_text = DOC_PATH.read_text(encoding='utf-8')
print(f'Read {len(raw_text)} characters from {DOC_PATH}.')


In [ ]:
# Header-Aware Chunking Algorithm
sections = re.split(r'\n(?=##? )', raw_text)
chunks = []

for idx, sec in enumerate(sections):
    sec = sec.strip()
    if not sec:
        continue
    
    words = sec.split()
    token_est = int(len(words) * 1.33)  # standard word to token ratio
    header_match = re.match(r'^(##?\s*[\w\s]+)', sec)
    title = header_match.group(1).replace('#', '').strip() if header_match else f'Section {idx+1}'
    
    chunk_id = hashlib.sha256(sec.encode('utf-8')).hexdigest()[:12]
    # Deterministic 16-dim normalized vector stub for offline vector indexing
    seed_val = int(chunk_id[:6], 16)
    embedding = [round(((seed_val >> i) & 15) / 15.0, 4) for i in range(16)]
    
    chunks.append({
        'chunk_id': f'chk_{chunk_id}',
        'title': title,
        'token_count': token_est,
        'text': sec,
        'embedding_dim': len(embedding),
        'embedding_preview': embedding[:4]
    })

with open(OUT_CHUNKS, 'w', encoding='utf-8') as f:
    json.dump({'document': 'documents.txt', 'total_chunks': len(chunks), 'chunks': chunks}, f, indent=2)
print(f'✅ Written: {OUT_CHUNKS} ({len(chunks)} chunks)')


In [ ]:
# Generate RAG Preparation Report
total_tokens = sum(c['token_count'] for c in chunks)
report_md = f'''# 📑 RAG Preparation & Chunking Report

**Generated:** {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}  
**Total Chunks Produced:** {len(chunks)}  
**Aggregate Tokens:** {total_tokens}  
**Embedding Dimensions:** 16 (Demonstration Vector / Ready for text-embedding-3-small)  

## Chunk Distribution Matrix

| Chunk ID | Section Title | Tokens | Sample Content Preview |
|---|---|---|---|
'''
for c in chunks:
    preview = c['text'].replace('\n', ' ')[:80] + '...'
    report_md += f"| `{c['chunk_id']}` | {c['title']} | {c['token_count']} | {preview} |\n"

report_md += '''
## Recommended Vector Store Ingestion
```python
# Insert into ChromaDB / pgvector / Pinecone:
# collection.add(
#     ids=[c['chunk_id'] for c in chunks],
#     documents=[c['text'] for c in chunks],
#     metadatas=[{'title': c['title']} for c in chunks]
# )
```
'''

with open(OUT_REPORT, 'w', encoding='utf-8') as f:
    f.write(report_md)
print(f'✅ Written: {OUT_REPORT}')


In [ ]:
# Smoke Test
assert OUT_CHUNKS.exists() and OUT_CHUNKS.stat().st_size > 0
assert OUT_REPORT.exists() and OUT_REPORT.stat().st_size > 0
print('✅ Pack 0006 execution completed successfully.')
